# Foreign Whispers - Colab GPU Backend Server

This notebook runs the FastAPI orchestrator directly in Google Colab. By running it natively here, the API will automatically fall back to its internal `faster-whisper` and `Coqui TTS` models, which will natively utilize the powerful Colab T4 GPU to process your long videos in seconds instead of hours!

**Instructions:**
1. Go to **Runtime > Change runtime type** and ensure **T4 GPU** is selected.
2. Run the Setup cell.
3. Paste your Ngrok token into the Server Boot cell and run it.
4. Copy the resulting Ngrok URL to your Mac's frontend configuration.

In [ ]:
# ==========================================
# 1. Setup & Installation (Run this first)
# ==========================================

!git clone https://github.com/aegean-ai/foreign-whispers.git
%cd foreign-whispers

# Install uv (fast python package manager) using pip so it is in the PATH
!pip install uv -q

# Install all project dependencies
!uv sync

# Install pyngrok into the global colab environment so we can import it
!pip install pyngrok nest-asyncio -q

In [ ]:
# ==========================================
# 2. Boot Server & Create Tunnel
# ==========================================

import subprocess
import time
import os
from pyngrok import ngrok

# Fix matplotlib backend error in Jupyter subprocesses
os.environ["MPLBACKEND"] = "Agg"

# --- PASTE YOUR NGROK TOKEN HERE ---
NGROK_TOKEN = "3D41N3dzj7hCpAfYDIXk2gdtNV1_3gBeNENvwHE32jporvsLR"
# -----------------------------------

ngrok.set_auth_token(NGROK_TOKEN)

# Start the FastAPI server in the background
print("Starting FastAPI Orchestrator...")
server_process = subprocess.Popen(
    ["uv", "run", "uvicorn", "api.src.main:app", "--host", "0.0.0.0", "--port", "8080"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    env=os.environ.copy()
)

time.sleep(5)  # Give uvicorn a few seconds to boot

# Open the Ngrok tunnel
public_url = ngrok.connect(8080).public_url
print("\n" + "="*60)
print("✅ SUCCESS! YOUR COLAB GPU BACKEND IS LIVE!")
print(f"🔥 Paste this URL into your Mac's frontend: {public_url}")
print("="*60 + "\n")

# Stream the server logs so you can monitor the pipeline progress
for line in iter(server_process.stdout.readline, ''):
    print(line, end='')
